# BioAI Hub — Clasificación de radiografías con PyTorch

Este notebook forma parte de un proyecto personal donde entreno modelos de visión por computadora para detectar neumonía en radiografías de tórax. Lo construí para aprender el flujo completo de ML: desde cargar datos hasta visualizar qué zonas activa el modelo con GradCAM.

El dataset base es el [Chest X-Ray de Kaggle](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) (NORMAL vs PNEUMONIA). En layers posteriores escalo a NIH ChestX-ray14 con 14 patologías y transfer learning.

---

**Layer 1 — Setup del entorno**

## Entorno de trabajo

Estoy usando Google Colab con GPU T4 (free tier). Colab es una máquina virtual que Google presta en la nube — tiene Python, CUDA y las librerías principales ya instaladas. Lo que no persiste entre sesiones es el disco `/content`, así que guardo checkpoints y datasets en Google Drive.

Para activar la GPU: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch

print(f"GPU disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Modelo: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB")
else:
    print("Sin GPU — revisar Runtime → Change runtime type")

## Google Drive

Monto Drive para que los checkpoints del modelo sobrevivan entre sesiones. Sin esto, si Colab resetea el runtime pierdo el entrenamiento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/drive/MyDrive/bioai-colab'
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Directorio de trabajo: {WORK_DIR}")
print(f"Contenido actual: {os.listdir(WORK_DIR)}")

In [ ]:
# scikit-learn, seaborn y tqdm no vienen en Colab por defecto
!pip install -q scikit-learn seaborn kaggle tqdm
print("ok")

## Tensores en PyTorch

Un tensor es el tipo de dato central en PyTorch. Funcionalmente es un array numérico, pero puede vivir en GPU y registra las operaciones que se hacen sobre él para calcular gradientes en el backward pass.

Las imágenes se representan con shape `[batch, canales, alto, ancho]`. Uso 224×224 porque es el tamaño estándar de la mayoría de los modelos pre-entrenados en ImageNet.

In [ ]:
# imagen simulada: 1 imagen, 3 canales, 224x224
x = torch.randn(1, 3, 224, 224)

print(f"Shape:  {x.shape}")
print(f"Device: {x.device}")
print(f"Dtype:  {x.dtype}")

In [ ]:
# mover a GPU
x = x.cuda()
print(f"Device: {x.device}")  # cuda:0

# las operaciones sobre tensores en GPU corren en los núcleos CUDA
# la diferencia de velocidad frente a CPU es significativa en redes grandes
print(f"Media: {x.mean():.4f}")
print(f"Std:   {x.std():.4f}")

## Verificación

Antes de avanzar a Layer 2 me aseguro de que el entorno esté completo.

In [ ]:
import torchvision, sklearn, pandas, matplotlib

print("Versiones:")
print(f"  torch:        {torch.__version__}")
print(f"  torchvision:  {torchvision.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  pandas:       {pandas.__version__}")
print(f"  matplotlib:   {matplotlib.__version__}")

print()
print("Estado:")
print(f"  GPU:   {'ok' if torch.cuda.is_available() else 'sin GPU'}")
print(f"  Drive: {'ok' if os.path.exists(WORK_DIR) else 'no montado'}")